In [2]:
path = '/mnt/c/Users/lucas/Desktop/Estudo/etl_eletronic_sales/data/electronics_sales_raw.csv'
import pandas as pd 


Neste notebook analisaremos pontos cruciais para o desenvolvimento financeira desta empresa fictícia de produtos eletônicos. Serão analisados aqui a quantidade de vendas por produto, região, subcategoria, vendedor, canal de vendas e tipo de cliente. O objetivo desta análise é garantir, através dos dados coletados, que a empresa esteja investindo corretamente seu capital no produto, canal e região certas para o seu desenvolvimento e, também, apontar pontos a melhorar para maximizar seus lucros.

In [3]:
# define o caminho do arquivo csv
df = pd.read_csv(path)
df.head()

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,discount_pct,sales_channel,payment_method,sales_rep,region,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag
0,10001,C1689,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,MX Keys S,Electronics,...,0.05,Online,Credit Card,Eduardo,South,239.83,14629.23,3092.03,329.57,0
1,10002,C1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,0.15,Online,Credit Card,Diego,North,81.55,8719.54,1189.63,88.62,1
2,10003,C1422,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,iPad mini,Electronics,...,0.15,Online,Cash,Camila,South,244.79,9475.50,3302.27,389.89,0
3,10004,C1308,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,0.00,Retail,Credit Card,Camila,West,304.91,12381.10,122.53,306.67,0
4,10005,C0826,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,0.05,Online,Credit Card,Eduardo,Central,24.90,12661.78,2031.60,317.36,0


In [4]:
# verifica se as colunas numéricas estão com o tipo correto, assim evitando erros na visualização e análise dos dados
numeric_columns = df.select_dtypes(include='number').columns
for column in numeric_columns:
    df[column] = df[column].fillna(0)
df.dtypes

order_id                 int64
customer_id                str
customer_name              str
customer_segment           str
customer_type              str
first_purchase_date        str
last_purchase_date         str
product_id                 str
product_name               str
category                   str
sub_category               str
brand                      str
order_date                 str
quantity                 int64
unit_price             float64
discount_pct           float64
sales_channel              str
payment_method             str
sales_rep                  str
region                     str
operating_expenses     float64
cash_balance           float64
debt_balance           float64
monthly_burn           float64
churn_flag               int64
dtype: object

In [5]:
# verifica se as colunas de data estão com o tipo correto
date_cols = ['first_purchase_date', 'last_purchase_date', 'order_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
df.dtypes

order_id                        int64
customer_id                       str
customer_name                     str
customer_segment                  str
customer_type                     str
first_purchase_date    datetime64[us]
last_purchase_date     datetime64[us]
product_id                        str
product_name                      str
category                          str
sub_category                      str
brand                             str
order_date             datetime64[us]
quantity                        int64
unit_price                    float64
discount_pct                  float64
sales_channel                     str
payment_method                    str
sales_rep                         str
region                            str
operating_expenses            float64
cash_balance                  float64
debt_balance                  float64
monthly_burn                  float64
churn_flag                      int64
dtype: object

In [6]:
# remove espaços em branco das colunas de texto
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

In [7]:
for col in date_cols:
    df[f'{col}_ano'] = df[col].dt.year
    df[f'{col}_mes'] = df[col].dt.month
    df[f'{col}_trimestre'] = df[col].dt.quarter


df.head()

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,churn_flag,first_purchase_date_ano,first_purchase_date_mes,first_purchase_date_trimestre,last_purchase_date_ano,last_purchase_date_mes,last_purchase_date_trimestre,order_date_ano,order_date_mes,order_date_trimestre
0,10001,C1689,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,MX Keys S,Electronics,...,0,2023,8,3,2024,4,2,2024,4,2
1,10002,C1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,1,2023,2,1,2025,7,3,2025,7,3
2,10003,C1422,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,iPad mini,Electronics,...,0,2023,3,1,2024,1,1,2024,1,1
3,10004,C1308,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,0,2023,10,4,2024,1,1,2024,1,1
4,10005,C0826,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,0,2023,11,4,2024,12,4,2024,12,4


In [8]:
# verificando a consistência dos dados, garantindo que cada subcategoria esteja associada a apenas uma categoria
df.groupby('sub_category')['category'].nunique()

sub_category
Audio          1
Laptops        1
Peripherals    1
Smartphones    1
Tablets        1
Name: category, dtype: int64

In [9]:
df['region'] = df['region'].str.lower().str.strip()
print(df['region'].unique())

<StringArray>
['south', 'north', 'west', 'central', 'east']
Length: 5, dtype: str


In [10]:
# verificando a quantidade de registros por categoria e subcategoria, analisando quais produtos são mais vendidos
sales_per_sub_category = df.groupby(['sub_category', 'product_name']).size().reset_index(name='count').sort_values(by='count', ascending=False)
print(sales_per_sub_category)

   sub_category        product_name  count
29  Smartphones    Galaxy S24 Ultra    237
30  Smartphones             Poco X6    235
33  Smartphones           iPhone 14    235
31  Smartphones       Redmi Note 13    230
34  Smartphones           iPhone 15    230
35  Smartphones      iPhone 15 Plus    224
28  Smartphones          Galaxy S24    222
27  Smartphones          Galaxy A55    204
32  Smartphones           Xiaomi 14    202
11      Laptops       Latitude 5440    166
5         Audio           SRS-XB100    161
16      Laptops        ThinkPad E14    161
17      Laptops              XPS 13    159
14      Laptops      MacBook Air M3    159
9       Laptops           IdeaPad 5    158
10      Laptops         Inspiron 15    155
7         Audio          WF-1000XM5    154
8         Audio          WH-1000XM5    153
3         Audio              Flip 6    153
4         Audio        HomePod mini    153
1         Audio         AirPods Pro    151
19  Peripherals       320K Keyboard    151
15      Lap

In [11]:
# verificando a quantidade de registros por região
sales_per_region =df.groupby('region')['order_id'].size().reset_index(name='count').sort_values(by='count', ascending=False)
sales_per_region

,region,count
2,north,1434
4,west,1419
0,central,1412
3,south,1377
1,east,1358


In [12]:
# verificando a quantidade de registros por representante de vendas e canal de vendas
print(df['sales_rep'].unique())
print(df['sales_channel'].unique())

<StringArray>
['Eduardo', 'Diego', 'Camila', 'Ana', 'Bruno']
Length: 5, dtype: str
<StringArray>
['Online', 'Retail']
Length: 2, dtype: str


In [13]:
# verificando a quantidade de registros por canal de vendas
sales_per_channel = df.groupby('sales_channel')['order_id'].size().reset_index(name='count')
sales_per_channel.sort_values(by='count', ascending=False)

,sales_channel,count
0,Online,5186
1,Retail,1814


In [14]:
# verificando a quantidade de registros por representante de vendas
sales_per_rep = df.groupby('sales_rep')['order_id'].size().reset_index(name='count')
sales_per_rep.sort_values(by='count', ascending=False)

,sales_rep,count
2,Camila,1434
0,Ana,1415
1,Bruno,1396
4,Eduardo,1381
3,Diego,1374


In [15]:
# verificando a quantidade de pedidos por representante de vendas e canal de vendas
sales_per_channel_rep = df.groupby(['sales_rep', 'sales_channel'])['order_id'].nunique().reset_index(name='order_count')
sales_per_channel_rep.sort_values(by='order_count', ascending=False)

,sales_rep,sales_channel,order_count
4,Camila,Online,1070
0,Ana,Online,1069
2,Bruno,Online,1056
8,Eduardo,Online,997
6,Diego,Online,994
9,Eduardo,Retail,384
7,Diego,Retail,380
5,Camila,Retail,364
1,Ana,Retail,346
3,Bruno,Retail,340
